In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

## Config

In [ ]:
import json
from pathlib import Path

DEVICE = "cuda"
MODEL_ID = "stabilityai/stable-diffusion-3-medium-diffusers"
NUM_STEPS = 28
GUIDANCE_SCALE = 7.0

N_STEPS = 28
N_BLOCKS = 24

ATTR_SEEDS = list(range(5))       # seeds per attribute prompt
CONCEPT_SEEDS = list(range(30))   # seeds for bare-concept prompt

PROMPT_DIR = Path("./prompt")
OUTPUT_DIR = Path("./output/multi_concept")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JSON_FILES = sorted(PROMPT_DIR.glob("*.json"))
print(f"found {len(JSON_FILES)} concept JSON files:")
for p in JSON_FILES:
    d = json.load(open(p))
    n_attr = sum(len(v) for v in d["simple_attribute_prompts"].values())
    print(f"  - {p.name}: concept=\"{d['concept']}\", {n_attr} attributes")

## Load Pipeline

In [ ]:
import torch
from diffusers import StableDiffusion3Pipeline

pipe = StableDiffusion3Pipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
).to(DEVICE)

## Hook & Capture

캡처한 hidden state는 disk(.pt)에 저장하지 않고, run 끝에 CPU fp16으로 옮긴 뒤 바로 running-sum에 누적하여 메모리를 절약한다.

In [ ]:
import sys, gc
sys.path.insert(0, "../mode_path")

from sd3_hook import patch_transformer, attach_step_tracker

# ── step tracker (한 번만 attach) ──
try:
    _already_attached
except NameError:
    step_counter = attach_step_tracker(pipe.transformer, on_step=lambda s: True)
    if step_counter is None:
        raise RuntimeError("step tracker already attached. Restart kernel.")
    _already_attached = True

    # ── capture buffer (한 run분만 유지, GPU에 쌓임) ──
    captured = {}           # {step_idx: {block_idx: tensor(4096, 1536) f16 on GPU}}
    _active = {"on": False}

    def on_capture(block_idx, L_minus_1, L_after_attn, L_output, batch_size):
        if not _active["on"]:
            return
        s = step_counter[0]
        captured.setdefault(s, {})[block_idx] = L_output[1].half().clone()

    patch_transformer(pipe.transformer, on_capture)

print("hook ready.")

## Run helpers

`run_once(prompt, seed)` 는 한 번의 inference 를 수행하고 captured 를 CPU fp16 list 형태로 반환한다. 결과는 disk 에 저장하지 않는다.

In [ ]:
def run_once(prompt_text, seed):
    """single generation → return captured as list[step][block] of CPU fp16 tensors.
    GPU buffer is cleared before return."""
    captured.clear()
    step_counter[0] = 0
    _active["on"] = True

    generator = torch.Generator(device=pipe.device).manual_seed(seed)
    image = pipe(
        prompt_text,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    ).images[0]
    _active["on"] = False

    # GPU → CPU fp16 즉시 이동 (GPU 메모리 해제)
    out = [[None]*N_BLOCKS for _ in range(N_STEPS)]
    for s in range(N_STEPS):
        for b in range(N_BLOCKS):
            out[s][b] = captured[s][b].detach().to("cpu", dtype=torch.float16).flatten()

    captured.clear()
    torch.cuda.empty_cache()
    gc.collect()
    return out, image


def accumulate_sum(accumulator, captured_cpu):
    """accumulator[s][b] (fp32 CPU) += captured_cpu[s][b] (fp16 CPU)"""
    for s in range(N_STEPS):
        for b in range(N_BLOCKS):
            v = captured_cpu[s][b].float()
            if accumulator[s][b] is None:
                accumulator[s][b] = v
            else:
                accumulator[s][b] += v


def cos_sim_heatmap(sum_A, sum_B):
    """cos(mean_A, mean_B) = cos(sum_A, sum_B) (scale invariant). → (N_STEPS, N_BLOCKS) float."""
    import numpy as np
    h = np.zeros((N_STEPS, N_BLOCKS), dtype=np.float32)
    for s in range(N_STEPS):
        for b in range(N_BLOCKS):
            a = sum_A[s][b]
            c = sum_B[s][b]
            dot = (a * c).sum().item()
            na = a.norm().item()
            nc = c.norm().item()
            h[s, b] = dot / max(na * nc, 1e-12)
    return h

## Per-JSON experiment

한 JSON 당:
1. concept prompt 를 `CONCEPT_SEEDS` 개만큼 돌려 `concept_sum` 누적 (CPU fp32).
2. 각 attribute 에 대해 `ATTR_SEEDS` 개 돌려 `attr_sum` 누적 → cos-sim heatmap (28×24) 계산 → attr_sum 해제.
3. concept_sum 해제.

메모리 피크: concept_sum(≈16.8 GB fp32) + 현재 attr_sum(≈16.8 GB fp32) ≈ 33.6 GB CPU.

In [ ]:
import time

def run_experiment_for_json(json_path, save_images=True):
    data = json.load(open(json_path))
    concept = data["concept"]
    attr_items = []  # list of (attr_type, attr_value, prompt_text)
    for at, d in data["simple_attribute_prompts"].items():
        for av, pt in d.items():
            attr_items.append((at, av, pt))

    print(f"\n=== {json_path.name} (concept={concept}, {len(attr_items)} attrs) ===")
    t0 = time.time()

    # image 저장은 옵션
    if save_images:
        img_root = OUTPUT_DIR / concept / "images"
        img_root.mkdir(parents=True, exist_ok=True)

    # ── 1) concept sum ──
    print(f"  [concept] running {len(CONCEPT_SEEDS)} seeds...")
    concept_sum = [[None]*N_BLOCKS for _ in range(N_STEPS)]
    for i, seed in enumerate(CONCEPT_SEEDS):
        captured_cpu, image = run_once(concept, seed)
        accumulate_sum(concept_sum, captured_cpu)
        if save_images:
            image.save(img_root / f"{concept}_seed{seed}.png")
        del captured_cpu, image
        if (i+1) % 10 == 0 or i+1 == len(CONCEPT_SEEDS):
            print(f"    concept {i+1}/{len(CONCEPT_SEEDS)}  ({time.time()-t0:.1f}s)")

    # ── 2) 각 attribute sum + cos sim ──
    heatmaps = {}  # {(attr_type, attr_value): (28,24) np.ndarray}
    for ai, (at, av, ptxt) in enumerate(attr_items):
        attr_sum = [[None]*N_BLOCKS for _ in range(N_STEPS)]
        for seed in ATTR_SEEDS:
            captured_cpu, image = run_once(ptxt, seed)
            accumulate_sum(attr_sum, captured_cpu)
            if save_images:
                image.save(img_root / f"{at}_{av}_seed{seed}.png")
            del captured_cpu, image

        hmap = cos_sim_heatmap(attr_sum, concept_sum)
        heatmaps[(at, av)] = hmap

        del attr_sum
        gc.collect()
        print(f"  [attr {ai+1}/{len(attr_items)}] {at}/{av}: sim range=[{hmap.min():.3f},{hmap.max():.3f}]  ({time.time()-t0:.1f}s)")

    del concept_sum
    gc.collect()

    return {
        "concept": concept,
        "json_file": json_path.name,
        "heatmaps": heatmaps,  # {(attr_type, attr_value): (28,24) ndarray}
    }

## Run across all JSON files

In [ ]:
# all_results: list of dicts, one per JSON
all_results = []
for jp in JSON_FILES:
    res = run_experiment_for_json(jp, save_images=True)
    all_results.append(res)

print(f"\n==> done. {len(all_results)} concepts processed.")
total_heatmaps = sum(len(r["heatmaps"]) for r in all_results)
print(f"==> total heatmaps collected: {total_heatmaps}")

## Aggregate & Plot

**두 단계 평균** (concept 동등 가중):
1. 한 JSON 내부 → 그 concept 의 attr heatmap 들을 평균 → concept 당 `(28,24)` 하나.
2. concept 별 heatmap 들을 평균 → 최종 `(28,24)`.

이렇게 하면 attr 개수가 많은 concept(예: couch 22개)이 적은 concept(예: car 13개)보다 과대평가되지 않음.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── 1단계: JSON(=concept) 내부에서 attr heatmap 평균 ──
per_concept_mean = {}   # concept -> (28, 24)
per_concept_meta = {}   # concept -> {n_attrs, json_file}
for r in all_results:
    hs = np.stack(list(r["heatmaps"].values()), axis=0)   # (n_attrs, 28, 24)
    per_concept_mean[r["concept"]] = hs.mean(axis=0)
    per_concept_meta[r["concept"]] = {
        "n_attrs": hs.shape[0],
        "json_file": r["json_file"],
    }

# ── 2단계: concept 별 평균들을 다시 평균 (concept 동등 가중) ──
concept_stack = np.stack(list(per_concept_mean.values()), axis=0)   # (n_concepts, 28, 24)
mean_hmap = concept_stack.mean(axis=0)
std_hmap  = concept_stack.std(axis=0)   # concept 간 편차

print(f"concepts aggregated: {len(per_concept_mean)}")
for c, info in per_concept_meta.items():
    print(f"  - {c}: avg of {info['n_attrs']} attrs  ({info['json_file']})")
print(f"final mean sim range: [{mean_hmap.min():.4f}, {mean_hmap.max():.4f}]")
print(f"across-concept std range: [{std_hmap.min():.4f}, {std_hmap.max():.4f}]")

# ── plot ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im0 = axes[0].imshow(mean_hmap, aspect="auto", cmap="RdYlBu_r", origin="lower")
axes[0].set_xlabel("block"); axes[0].set_ylabel("step")
axes[0].set_title(f"Mean cos sim (attr vs concept)\n"
                  f"(per-concept avg over attrs) → avg over {len(per_concept_mean)} concepts")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(std_hmap, aspect="auto", cmap="viridis", origin="lower")
axes[1].set_xlabel("block"); axes[1].set_ylabel("step")
axes[1].set_title(f"Std across {len(per_concept_mean)} concepts")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mean_cossim_allprompts.png", dpi=130)
plt.show()

### Per-concept breakdown (각 JSON 별 평균 heatmap)

In [ ]:
n = len(per_concept_mean)
cols = 3
rows = (n + cols - 1) // cols

all_vals = np.concatenate([m.ravel() for m in per_concept_mean.values()])
vmin, vmax = all_vals.min(), all_vals.max()

fig, axes = plt.subplots(rows, cols, figsize=(cols*4.5, rows*3.8))
axes_flat = axes.flat if rows*cols > 1 else [axes]

for i, (concept, m) in enumerate(per_concept_mean.items()):
    ax = axes_flat[i]
    im = ax.imshow(m, aspect="auto", cmap="RdYlBu_r", origin="lower", vmin=vmin, vmax=vmax)
    na = per_concept_meta[concept]["n_attrs"]
    ax.set_title(f"{concept}  (avg of {na} attrs)", fontsize=10)
    ax.set_xlabel("block"); ax.set_ylabel("step")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for j in range(len(per_concept_mean), rows*cols):
    axes_flat[j].axis("off")

fig.suptitle("Per-concept mean cosine similarity (attr vs concept)", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_concept_cossim.png", dpi=130)
plt.show()

### Per-attribute-type breakdown (color / state / shape / ...)

In [ ]:
from collections import defaultdict

# (concept, attr_type, attr_value) → heatmap 평면화
by_type = defaultdict(list)
for r in all_results:
    for (at, av), h in r["heatmaps"].items():
        by_type[at].append(h)

# 공통 color scale
type_means = {at: np.stack(hs, axis=0).mean(axis=0) for at, hs in by_type.items()}
all_vals = np.concatenate([m.ravel() for m in type_means.values()])
vmin, vmax = all_vals.min(), all_vals.max()

n = len(type_means)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols*4.5, rows*3.8))
axes_flat = axes.flat if rows*cols > 1 else [axes]

for i, (at, m) in enumerate(type_means.items()):
    ax = axes_flat[i]
    im = ax.imshow(m, aspect="auto", cmap="RdYlBu_r", origin="lower", vmin=vmin, vmax=vmax)
    ax.set_title(f"{at}  (n={len(by_type[at])})", fontsize=10)
    ax.set_xlabel("block"); ax.set_ylabel("step")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for j in range(len(type_means), rows*cols):
    axes_flat[j].axis("off")

fig.suptitle("Per-attribute-type mean cosine similarity", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_attrtype_cossim.png", dpi=130)
plt.show()

### Save aggregated heatmaps (numpy, 가벼움)

In [ ]:
# heatmap 들은 28×24 작은 배열이므로 .npz 로 저장
save_dict = {}
for r in all_results:
    for (at, av), h in r["heatmaps"].items():
        key = f"{r['concept']}__{at}__{av}"
        save_dict[key] = h
for c, m in per_concept_mean.items():
    save_dict[f"__concept_mean__{c}"] = m
save_dict["__mean_all__"] = mean_hmap
save_dict["__std_all__"] = std_hmap

np.savez_compressed(OUTPUT_DIR / "heatmaps.npz", **save_dict)
print(f"saved {len(save_dict)} arrays → {OUTPUT_DIR / 'heatmaps.npz'}")